In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip "/content/drive/MyDrive/archive.zip"

In [ ]:
import os
import json
import torch
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

from tqdm import tqdm

from torch import nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

In [ ]:
train_data = []

with open('/content/train_data.json', 'r') as f:

    for line in f:

        train_data.append(json.loads(line))

val_data = []

with open('/content/val_data.json', 'r') as f:

    for line in f:

        val_data.append(json.loads(line))

In [ ]:
labels = sorted(
    list(set(item['class_label'] for item in train_data))
)

label_to_idx = {
    label: idx for idx, label in enumerate(labels)
}

idx_to_label = {
    idx: label for label, idx in label_to_idx.items()
}

num_classes = len(labels)

print(num_classes)

In [ ]:
train_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ToTensor()
])

val_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor()
])

In [ ]:
class IndoFashionDataset(Dataset):

    def __init__(
        self,
        data,
        label_to_idx,
        transform=None
    ):

        self.data = data

        self.label_to_idx = label_to_idx

        self.transform = transform

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        item = self.data[idx]

        image = Image.open(
            item['image_path']
        ).convert('RGB')

        label = self.label_to_idx[
            item['class_label']
        ]

        if self.transform:

            image = self.transform(image)

        return image, label

In [ ]:
train_dataset = IndoFashionDataset(
    train_data,
    label_to_idx,
    train_transform
)

val_dataset = IndoFashionDataset(
    val_data,
    label_to_idx,
    val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
import os

files = os.listdir('/content/drive/MyDrive')

for file in files:

    if "efficient" in file.lower():

        print(file)

In [ ]:
import os

folder_path = "/content/drive/MyDrive/EfficientNet_Model"

print(os.listdir(folder_path))

In [ ]:
from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights
)

model = efficientnet_b0(
    weights=EfficientNet_B0_Weights.DEFAULT
)

num_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    num_features,
    num_classes
)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    patience=2
)

In [ ]:
scaler = torch.cuda.amp.GradScaler()

In [ ]:
train_accuracies = []
val_accuracies = []

train_losses = []
val_losses = []

best_val_accuracy = 0

epochs = 7

for epoch in range(epochs):

    model.train()

    train_loss = 0.0

    train_correct = 0
    train_total = 0

    loop = tqdm(train_loader)

    for images, labels_batch in loop:

        images = images.to(device)

        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():

            outputs = model(images)

            loss = criterion(
                outputs,
                labels_batch
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        train_total += labels_batch.size(0)

        train_correct += (
            predicted == labels_batch
        ).sum().item()

        loop.set_description(
            f"Epoch {epoch+1}"
        )

        loop.set_postfix(
            accuracy=100 * train_correct / train_total,
            loss=loss.item()
        )

    train_accuracy = (
        100 * train_correct / train_total
    )

    avg_train_loss = (
        train_loss / len(train_loader)
    )

    model.eval()

    val_correct = 0
    val_total = 0

    val_loss = 0.0

    with torch.no_grad():

        for images, labels_batch in val_loader:

            images = images.to(device)

            labels_batch = labels_batch.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels_batch
            )

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels_batch.size(0)

            val_correct += (
                predicted == labels_batch
            ).sum().item()

    validation_accuracy = (
        100 * val_correct / val_total
    )

    avg_val_loss = (
        val_loss / len(val_loader)
    )

    train_accuracies.append(
        train_accuracy
    )

    val_accuracies.append(
        validation_accuracy
    )

    train_losses.append(
        avg_train_loss
    )

    val_losses.append(
        avg_val_loss
    )

    print(f"\nEpoch {epoch+1} Completed")

    print(f"Train Loss: {avg_train_loss:.4f}")

    print(f"Train Accuracy: {train_accuracy:.2f}%")

    print(f"Validation Loss: {avg_val_loss:.4f}")

    print(f"Validation Accuracy: {validation_accuracy:.2f}%")

    torch.save(
        model.state_dict(),
        f"/content/drive/MyDrive/efficientnet_epoch_{epoch+1}.pth"
    )

    if validation_accuracy > best_val_accuracy:

        best_val_accuracy = validation_accuracy

        torch.save(
            model.state_dict(),
            "/content/drive/MyDrive/best_efficientnet_model.pth"
        )

        print("Best model saved!")

    scheduler.step(validation_accuracy)

In [ ]:
model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/best_efficientnet_model.pth",
        map_location=device
    )
)

model.train()

train_loss = 0.0

train_correct = 0

train_total = 0

loop = tqdm(train_loader)

for images, labels_batch in loop:

    images = images.to(device)

    labels_batch = labels_batch.to(device)

    optimizer.zero_grad()

    with torch.cuda.amp.autocast():

        outputs = model(images)

        loss = criterion(
            outputs,
            labels_batch
        )

    scaler.scale(loss).backward()

    scaler.step(optimizer)

    scaler.update()

    train_loss += loss.item()

    _, predicted = torch.max(
        outputs,
        1
    )

    train_total += labels_batch.size(0)

    train_correct += (
        predicted == labels_batch
    ).sum().item()

    loop.set_description("Epoch 8")

    loop.set_postfix(

        accuracy=
        100 * train_correct / train_total,

        loss=loss.item()
    )

train_accuracy = (
    100 * train_correct / train_total
)

avg_train_loss = (
    train_loss / len(train_loader)
)

model.eval()

val_correct = 0

val_total = 0

val_loss = 0.0

with torch.no_grad():

    for images, labels_batch in val_loader:

        images = images.to(device)

        labels_batch = labels_batch.to(device)

        outputs = model(images)

        loss = criterion(
            outputs,
            labels_batch
        )

        val_loss += loss.item()

        _, predicted = torch.max(
            outputs,
            1
        )

        val_total += labels_batch.size(0)

        val_correct += (
            predicted == labels_batch
        ).sum().item()

validation_accuracy = (
    100 * val_correct / val_total
)

avg_val_loss = (
    val_loss / len(val_loader)
)

print("\nEpoch 8 Completed")

print(
    f"Train Loss: {avg_train_loss:.4f}"
)

print(
    f"Train Accuracy: {train_accuracy:.2f}%"
)

print(
    f"Validation Loss: {avg_val_loss:.4f}"
)

print(
    f"Validation Accuracy: {validation_accuracy:.2f}%"
)

torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/efficientnet_epoch_8.pth"
)

print("Epoch 8 checkpoint saved!")

scheduler.step(validation_accuracy)

In [ ]:
torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/indo_fashion_best_efficientnet.pth"
)

print("EfficientNet Model Saved Successfully")

In [ ]:
torch.save({

    'model_state_dict': model.state_dict(),

    'optimizer_state_dict': optimizer.state_dict()

},

"/content/drive/MyDrive/indo_fashion_full_checkpoint.pth")

print("Full Checkpoint Saved")

In [ ]:
import os

files = os.listdir('/content/drive/MyDrive')

for file in files:

    if "efficientnet" in file.lower():

        print(file)

In [ ]:
train_accuracies = [
    83.42,
    87.38,
    88.56,
    89.31,
    89.89,
    90.43,
    90.88,
    91.05
]

val_accuracies = [
    84.75,
    87.33,
    86.73,
    87.77,
    88.61,
    88.89,
    89.21,
    89.09
]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(
    train_accuracies,
    label='Train Accuracy'
)

plt.plot(
    val_accuracies,
    label='Validation Accuracy'
)

plt.xlabel("Epochs")

plt.ylabel("Accuracy")

plt.title("EfficientNet Accuracy Graph")

plt.legend()

plt.savefig(
    "/content/drive/MyDrive/efficientnet_accuracy_graph.png"
)

plt.show()

In [ ]:
train_losses = [
    0.5323,
    0.4080,
    0.3688,
    0.3413,
    0.3206,
    0.3029,
    0.2860,
    0.2758
]

val_losses = [
    0.4557,
    0.3962,
    0.4092,
    0.3763,
    0.3623,
    0.3727,
    0.3371,
    0.3376
]

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    train_losses,
    label='Train Loss'
)

plt.plot(
    val_losses,
    label='Validation Loss'
)

plt.xlabel("Epochs")

plt.ylabel("Loss")

plt.title("EfficientNet Loss Graph")

plt.legend()

plt.savefig(
    "/content/drive/MyDrive/efficientnet_loss_graph.png"
)

plt.show()

In [ ]:
model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/best_efficientnet_model.pth",
        map_location=device
    )
)

model.eval()

print("Best EfficientNet Loaded")

In [ ]:
test_data = []

with open('/content/test_data.json', 'r') as f:

    for line in f:

        test_data.append(json.loads(line))

In [ ]:
test_dataset = IndoFashionDataset(
    test_data,
    label_to_idx,
    val_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
all_preds = []
all_labels = []

model.eval()

with torch.no_grad():

    for images, labels_batch in test_loader:

        images = images.to(device)

        labels_batch = labels_batch.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            labels_batch.cpu().numpy()
        )

In [ ]:
from sklearn.metrics import accuracy_score

test_accuracy = accuracy_score(
    all_labels,
    all_preds
)

print(
    f"Test Accuracy: {test_accuracy * 100:.2f}%"
)

In [ ]:
from sklearn.metrics import classification_report

import pandas as pd

import matplotlib.pyplot as plt

report = classification_report(
    all_labels,
    all_preds,
    output_dict=True
)

report_df = pd.DataFrame(
    report
).transpose()

plt.figure(figsize=(12,8))

plt.axis('off')

table = plt.table(

    cellText=report_df.round(2).values,

    colLabels=report_df.columns,

    rowLabels=report_df.index,

    loc='center'
)

table.auto_set_font_size(False)

table.set_fontsize(10)

table.scale(1.2, 1.2)

plt.title(
    "EfficientNet Classification Report",
    pad=20
)

plt.savefig(
    "/content/drive/MyDrive/efficientnet_classification_report.png",
    bbox_inches='tight'
)

plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix

import seaborn as sns

import matplotlib.pyplot as plt

cm = confusion_matrix(
    all_labels,
    all_preds
)

plt.figure(figsize=(14,12))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted Labels")

plt.ylabel("True Labels")

plt.title("EfficientNet Confusion Matrix")

plt.savefig(
    "/content/drive/MyDrive/efficientnet_confusion_matrix.png"
)

plt.show()